# MMM LBE Diagnostics from Saved Exports

This notebook recreates an EXH-style diagnostics view for [MMM_LBE_Model_Training](#notebook-3390981627619814) using only saved exports.

What this notebook does:
* Loads the saved `lbe_mmm_results.pkl` artifact and existing output files
* Reconstructs validation, residual, posterior, adstock, decomposition, and counterfactual charts
* Saves new charts to a separate `exports_only_diagnostics` folder so nothing existing is overwritten

What this notebook does not do:
* No model retraining
* No resampling or posterior refit
* No overwrite of existing LBE output files


In [0]:
%pip install xarray arviz scipy openpyxl --quiet

In [0]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.stats import beta as beta_dist, halfnorm, norm

RESULTS_PKL = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_LBE/lbe_mmm_results.pkl"
OUTPUT_DIR = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_LBE"
EXPORT_DIR = f"{OUTPUT_DIR}/exports_only_diagnostics"
os.makedirs(EXPORT_DIR, exist_ok=True)

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#F0F2F5"
CPX_GREEN = "#2E8B57"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#CCCCCC",
    "grid.color": "#EEEEEE",
    "axes.grid": True,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})


def save_fig(fig, filename):
    path = os.path.join(EXPORT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")
    return path


In [0]:
with open(RESULTS_PKL, "rb") as f:
    results = pickle.load(f)

idata = results["idata"]
draws = results["draws"]
bc = results["bc"]
n_draws = int(results["n_draws"])
tr = results["tr"]
va = results["va"]
df_train = results["df_train"].copy()
df_val = results["df_val"].copy()
y_mean = float(results["y_mean"])
y_std = float(results["y_std"])
aw_max = float(results["aw_max"])
co_max = float(results["co_max"])
t_max = float(results["t_max"])
temp_mu = float(results["temp_mu"])
temp_sd = float(results["temp_sd"])
cal_cols = list(results["CAL_COLS"])
metrics = results["metrics"]
decomp_rows = results["decomp"]
counterfactuals = results["counterfactuals"]

for frame, segment in [(df_train, "train"), (df_val, "val")]:
    frame["segment"] = segment
    frame["week_date"] = pd.to_datetime(frame["week_date"])

df_all = pd.concat([df_train, df_val], ignore_index=True).sort_values("week_date").reset_index(drop=True)

full_data = {
    "aw": np.concatenate([np.asarray(tr["aw"]), np.asarray(va["aw"])]),
    "co": np.concatenate([np.asarray(tr["co"]), np.asarray(va["co"])]),
    "t": np.concatenate([np.asarray(tr["t"]), np.asarray(va["t"])]),
    "temp": np.concatenate([np.asarray(tr["temp"]), np.asarray(va["temp"])]),
    "cal": np.vstack([np.asarray(tr["cal"]), np.asarray(va["cal"])]),
}

files_df = pd.DataFrame([
    {
        "file_name": name,
        "size_kb": round(os.path.getsize(os.path.join(OUTPUT_DIR, name)) / 1024, 1)
    }
    for name in sorted(os.listdir(OUTPUT_DIR))
    if os.path.isfile(os.path.join(OUTPUT_DIR, name))
])

summary_df = pd.DataFrame([
    {"metric": "Train MAPE", "value": round(float(metrics["train_mape"]), 2)},
    {"metric": "Validation MAPE", "value": round(float(metrics["val_mape"]), 2)},
    {"metric": "Train R2", "value": round(float(metrics["train_r2"]), 3)},
    {"metric": "Validation R2", "value": round(float(metrics["val_r2"]), 3)},
    {"metric": "Posterior draws", "value": n_draws},
    {"metric": "Train weeks", "value": len(df_train)},
    {"metric": "Validation weeks", "value": len(df_val)},
])

print("Loaded saved LBE exports only; no retraining performed.")
display(summary_df)
display(files_df)


In [0]:
def adstock_np(x: np.ndarray, decay: float, max_lag: int = 10) -> np.ndarray:
    w = decay ** np.arange(max_lag)
    w = w / w.sum()
    out = np.zeros(len(x), dtype=float)
    for lag in range(max_lag):
        out[lag:] += w[lag] * x[: len(x) - lag]
    return out


def predict_revenue(data: dict, sample_size: int = 1000, seed: int = 42) -> np.ndarray:
    sample_size = min(sample_size, n_draws)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n_draws, sample_size, replace=False)
    preds = np.zeros((sample_size, len(data["aw"])), dtype=float)
    for j, i in enumerate(idx):
        aw_ads = adstock_np(np.asarray(data["aw"]), float(draws["aw_decay"][i]))
        co_ads = adstock_np(np.asarray(data["co"]), float(draws["co_decay"][i]))
        aw_sat = np.log1p(aw_ads * float(draws["aw_scale"][i]))
        co_sat = np.log1p(co_ads * float(draws["co_scale"][i]))
        cal_eff = np.asarray(data["cal"]) @ bc[i]
        mu = (
            float(draws["intercept"][i])
            + float(draws["beta_aw"][i]) * aw_sat
            + float(draws["beta_co"][i]) * co_sat
            + float(draws["beta_trend"][i]) * np.asarray(data["t"])
            + float(draws["beta_temp"][i]) * np.asarray(data["temp"])
            + cal_eff
        )
        preds[j] = np.exp(mu * y_std + y_mean)
    return preds


def decompose_revenue(data: dict, sample_size: int = 800, seed: int = 42):
    sample_size = min(sample_size, n_draws)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n_draws, sample_size, replace=False)
    t_len = len(data["aw"])
    baseline = np.zeros((sample_size, t_len), dtype=float)
    aw_contrib = np.zeros((sample_size, t_len), dtype=float)
    co_contrib = np.zeros((sample_size, t_len), dtype=float)
    total_pred = np.zeros((sample_size, t_len), dtype=float)

    for j, i in enumerate(idx):
        aw_ads = adstock_np(np.asarray(data["aw"]), float(draws["aw_decay"][i]))
        co_ads = adstock_np(np.asarray(data["co"]), float(draws["co_decay"][i]))
        aw_sat = np.log1p(aw_ads * float(draws["aw_scale"][i]))
        co_sat = np.log1p(co_ads * float(draws["co_scale"][i]))
        cal_eff = np.asarray(data["cal"]) @ bc[i]

        base_log = (
            float(draws["intercept"][i])
            + float(draws["beta_trend"][i]) * np.asarray(data["t"])
            + float(draws["beta_temp"][i]) * np.asarray(data["temp"])
            + cal_eff
        )
        aw_log = float(draws["beta_aw"][i]) * aw_sat
        co_log = float(draws["beta_co"][i]) * co_sat
        total_log = base_log + aw_log + co_log

        baseline[j] = np.exp(base_log * y_std + y_mean)
        aw_contrib[j] = np.exp(total_log * y_std + y_mean) - np.exp((base_log + co_log) * y_std + y_mean)
        co_contrib[j] = np.exp(total_log * y_std + y_mean) - np.exp((base_log + aw_log) * y_std + y_mean)
        total_pred[j] = np.exp(total_log * y_std + y_mean)

    return {
        "baseline_mean": baseline.mean(axis=0),
        "aw_mean": aw_contrib.mean(axis=0),
        "co_mean": co_contrib.mean(axis=0),
        "pred_mean": total_pred.mean(axis=0),
    }


train_preds = predict_revenue(tr, sample_size=1000, seed=42)
val_preds = predict_revenue(va, sample_size=1000, seed=43)
full_preds = predict_revenue(full_data, sample_size=1000, seed=44)

train_actual = np.exp(np.asarray(tr["y"]) * y_std + y_mean)
val_actual = np.exp(np.asarray(va["y"]) * y_std + y_mean)
full_actual = df_all["LBE_Revenue"].to_numpy()

train_mean = train_preds.mean(axis=0)
val_mean = val_preds.mean(axis=0)
full_mean = full_preds.mean(axis=0)

train_resid = train_actual - train_mean
val_resid = val_actual - val_mean

attrib = decompose_revenue(full_data, sample_size=800, seed=45)

posterior_summary = pd.DataFrame([
    {"parameter": "aw_decay", "mean": draws["aw_decay"].mean(), "p2.5": np.percentile(draws["aw_decay"], 2.5), "p97.5": np.percentile(draws["aw_decay"], 97.5)},
    {"parameter": "co_decay", "mean": draws["co_decay"].mean(), "p2.5": np.percentile(draws["co_decay"], 2.5), "p97.5": np.percentile(draws["co_decay"], 97.5)},
    {"parameter": "beta_aw",  "mean": draws["beta_aw"].mean(),  "p2.5": np.percentile(draws["beta_aw"], 2.5),  "p97.5": np.percentile(draws["beta_aw"], 97.5)},
    {"parameter": "beta_co",  "mean": draws["beta_co"].mean(),  "p2.5": np.percentile(draws["beta_co"], 2.5),  "p97.5": np.percentile(draws["beta_co"], 97.5)},
])

display(posterior_summary)


In [0]:
prior_specs = [
    ("aw_decay",  "Awareness decay",      np.linspace(0, 1, 400), lambda x: beta_dist(a=6.5, b=3.5).pdf(x), CPX_NAVY),
    ("co_decay",  "Consideration decay",  np.linspace(0, 1, 400), lambda x: beta_dist(a=3.0, b=7.0).pdf(x), CPX_GOLD),
    ("aw_scale",  "Awareness scale",      np.linspace(0, 5, 400), lambda x: halfnorm(scale=2.0).pdf(x), CPX_NAVY),
    ("co_scale",  "Consideration scale",  np.linspace(0, 3, 400), lambda x: halfnorm(scale=1.0).pdf(x), CPX_GOLD),
    ("beta_aw",   "Awareness coefficient", np.linspace(0, 2, 400), lambda x: halfnorm(scale=0.8).pdf(x), CPX_NAVY),
    ("beta_co",   "Consideration coefficient", np.linspace(0, 1, 400), lambda x: halfnorm(scale=0.3).pdf(x), CPX_GOLD),
    ("beta_trend","Trend coefficient",    np.linspace(-0.3, 0.8, 400), lambda x: norm(loc=0.2, scale=0.2).pdf(x), CPX_RED),
    ("beta_temp", "Temperature coefficient", np.linspace(-0.8, 0.4, 400), lambda x: norm(loc=-0.2, scale=0.15).pdf(x), CPX_RED),
    ("sigma",     "Residual sigma",       np.linspace(0, 0.8, 400), lambda x: halfnorm(scale=0.3).pdf(x), CPX_DGREY),
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()
for ax, (param, label, x_grid, prior_fn, color) in zip(axes, prior_specs):
    vals = np.asarray(draws[param])
    ax.hist(vals, bins=35, density=True, color=color, alpha=0.35, edgecolor="white")
    ax.plot(x_grid, prior_fn(x_grid), color=CPX_RED, linewidth=2, label="Prior")
    ax.axvline(vals.mean(), color=color, linestyle="--", linewidth=1.5, label="Posterior mean")
    ax.set_title(label)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle("LBE MMM — Prior vs Posterior From Saved Exports", fontsize=14, fontweight="bold", color=CPX_NAVY)
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, "lbe_d1_prior_vs_posterior_exports_only.png")
plt.show()

weeks_out = 16
weeks = np.arange(weeks_out)
fig, ax = plt.subplots(figsize=(10, 5))
for name, key, color in [("Awareness", "aw_decay", CPX_NAVY), ("Consideration", "co_decay", CPX_GOLD)]:
    vals = np.asarray(draws[key])
    mean_decay = np.array([vals.mean() ** t for t in weeks])
    lo_decay = np.array([np.quantile(vals, 0.025) ** t for t in weeks])
    hi_decay = np.array([np.quantile(vals, 0.975) ** t for t in weeks])
    ax.plot(weeks, mean_decay, color=color, linewidth=2, label=f"{name} (mean={vals.mean():.2f})")
    ax.fill_between(weeks, lo_decay, hi_decay, color=color, alpha=0.18, label="_nolegend_")
ax.axhline(0.05, color=CPX_DGREY, linestyle="--", linewidth=1, alpha=0.8)
ax.set_xlabel("Weeks after spend")
ax.set_ylabel("Remaining effect")
ax.set_title("LBE MMM — Adstock Decay Curves From Posterior Exports\nShaded = 95% credible interval  |  Dashed = 5% threshold", color=CPX_NAVY)
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_d7_adstock_decay_exports_only.png")
plt.show()


In [0]:
# ── D3: Actual vs Predicted (full series) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(df_all["week_date"], full_actual / 1e6, color="black", linewidth=1.5, label="Actual")
ax.plot(df_all["week_date"], full_mean / 1e6, color=CPX_NAVY, linewidth=1.8, linestyle="--", label="Predicted")
ax.axvspan(df_val["week_date"].min(), df_val["week_date"].max(), alpha=0.08, color=CPX_RED, label="Validation window")
ax.set_title(
    f"LBE MMM — Actual vs Predicted\n"
    f"Train MAPE {metrics['train_mape']:.1f}%  R² {metrics['train_r2']:.3f}  |  "
    f"Val MAPE {metrics['val_mape']:.1f}%  R² {metrics['val_r2']:.3f}",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.set_ylabel("Weekly Revenue ($M)")
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_d3_actual_vs_predicted.png")
plt.show()

# ── D4: Validation Window Detail ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(df_val["week_date"], val_actual / 1e6, color=CPX_NAVY, linewidth=2, label="Actual")
ax.plot(df_val["week_date"], val_mean / 1e6, color=CPX_GOLD, linewidth=2, linestyle="--", label="Predicted")
ax.fill_between(df_val["week_date"], val_actual / 1e6, val_mean / 1e6,
                where=(val_actual >= val_mean), color=CPX_GREEN, alpha=0.22, label="Under-predicted")
ax.fill_between(df_val["week_date"], val_actual / 1e6, val_mean / 1e6,
                where=(val_actual < val_mean), color=CPX_RED, alpha=0.18, label="Over-predicted")
ax.set_title(
    f"LBE MMM — Validation Window Detail\n"
    f"Val MAPE {metrics['val_mape']:.1f}%  R² {metrics['val_r2']:.3f}",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.set_ylabel("Weekly Revenue ($M)")
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_d4_validation_window.png")
plt.show()

# ── D5: Residuals Over Time ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
resid_band = np.std(train_resid / 1e6)
ax.plot(df_train["week_date"], train_resid / 1e6, color=CPX_NAVY, linewidth=1.2, label="Train residual")
ax.bar(df_val["week_date"], val_resid / 1e6, color=CPX_RED, alpha=0.70, width=5, label="Validation residual")
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline( resid_band, color=CPX_DGREY, linestyle="--", linewidth=1,
            label=f"+1σ train ({resid_band:+.2f}M)")
ax.axhline(-resid_band, color=CPX_DGREY, linestyle="--", linewidth=1,
            label=f"-1σ train ({-resid_band:+.2f}M)")
ax.set_title(
    "LBE MMM — Residuals Over Time\nTarget: random, no structure",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.set_ylabel("Residual ($M)")
ax.legend(frameon=False, ncol=2)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_d5_residuals_over_time.png")
plt.show()

# ── D6: Predicted vs Actual Scatter ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.scatter(train_actual / 1e6, train_mean / 1e6, color=CPX_NAVY, alpha=0.55, s=35, label="Train")
ax.scatter(val_actual / 1e6, val_mean / 1e6, color=CPX_RED, alpha=0.90, s=70, marker="D", label="Validation")
lo = min(train_actual.min(), val_actual.min(), train_mean.min(), val_mean.min()) / 1e6
hi = max(train_actual.max(), val_actual.max(), train_mean.max(), val_mean.max()) / 1e6
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, label="Perfect fit")
ax.set_xlabel("Actual Revenue ($M)")
ax.set_ylabel("Predicted Revenue ($M)")
ax.set_title(
    f"LBE MMM — Predicted vs Actual Scatter\n"
    f"Train R² {metrics['train_r2']:.3f}  |  Val R² {metrics['val_r2']:.3f}",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_d6_scatter_predicted_vs_actual.png")
plt.show()


In [0]:
years = [row["year"] for row in decomp_rows]
base_vals = np.array([row["baseline"] for row in decomp_rows]) / 1e6
aw_vals = np.array([row["awareness"] for row in decomp_rows]) / 1e6
co_vals = np.array([row["consideration"] for row in decomp_rows]) / 1e6
actual_vals = np.array([row["actual"] for row in decomp_rows]) / 1e6
aw_roi = np.array([row["aw_roi"] for row in decomp_rows])
co_roi = np.array([row["co_roi"] for row in decomp_rows])
# Recompute at 95% CI from raw per-draw arrays stored in decomp_rows
aw_roi_lo = np.array([np.percentile(row["aw_r"] / row["aw_sp"], 2.5) for row in decomp_rows])
aw_roi_hi = np.array([np.percentile(row["aw_r"] / row["aw_sp"], 97.5) for row in decomp_rows])
co_roi_lo = np.array([np.percentile(row["co_r"] / row["co_sp"], 2.5) for row in decomp_rows])
co_roi_hi = np.array([np.percentile(row["co_r"] / row["co_sp"], 97.5) for row in decomp_rows])

cf_df = pd.DataFrame([
    {
        "scenario": name,
        "mean_revenue_m": vals["mean"] / 1e6,
        "ci_lo_m": vals["ci80_lo"] / 1e6,
        "ci_hi_m": vals["ci80_hi"] / 1e6,
    }
    for name, vals in counterfactuals.items()
])
actual_cf = float(cf_df.loc[cf_df["scenario"] == "Actual", "mean_revenue_m"].iloc[0])
cf_df["delta_vs_actual_m"] = cf_df["mean_revenue_m"] - actual_cf

x = np.arange(len(years))

# ── B1: Weekly Revenue Decomposition ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
stack_bottom = attrib["baseline_mean"] / 1e6
ax.fill_between(df_all["week_date"], 0, stack_bottom, color=CPX_LGREY, alpha=0.95, label="Baseline")
ax.fill_between(df_all["week_date"], stack_bottom, stack_bottom + attrib["aw_mean"] / 1e6,
                color=CPX_NAVY, alpha=0.80, label="Awareness")
ax.fill_between(df_all["week_date"],
                stack_bottom + attrib["aw_mean"] / 1e6,
                stack_bottom + attrib["aw_mean"] / 1e6 + attrib["co_mean"] / 1e6,
                color=CPX_GOLD, alpha=0.80, label="Consideration")
ax.plot(df_all["week_date"], full_actual / 1e6, color="black", linewidth=1.5, label="Actual")
ax.axvspan(df_val["week_date"].min(), df_val["week_date"].max(), alpha=0.08, color=CPX_RED, label="Validation")
ax.set_title("LBE MMM — Weekly Revenue Decomposition", fontsize=13, fontweight="bold", color=CPX_NAVY)
ax.set_ylabel("Revenue ($M)")
ax.legend(frameon=False, ncol=5, loc="upper left")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_b1_weekly_decomposition.png")
plt.show()

# ── B6: Annual Revenue Attribution ─────────────────────────────────────────
# Scale model contributions proportionally so bars sum exactly to actuals.
# Each segment share = model_contribution / model_total; scaled = share × actual
model_totals = base_vals + aw_vals + co_vals
base_scaled  = base_vals / model_totals * actual_vals
aw_scaled    = aw_vals   / model_totals * actual_vals
co_scaled    = co_vals   / model_totals * actual_vals

fig, ax = plt.subplots(figsize=(16, 9))
w = 0.55

ax.bar(x, base_scaled,                       width=w, color=CPX_LGREY, edgecolor="#BBBBBB",
       label="Baseline (seasonality + weather + trend)")
ax.bar(x, aw_scaled,   bottom=base_scaled,   width=w, color=CPX_NAVY, alpha=0.85,
       label="Awareness media contribution")
ax.bar(x, co_scaled,   bottom=base_scaled + aw_scaled, width=w, color=CPX_GOLD, alpha=0.85,
       label="Consideration media contribution")

for i in range(len(years)):
    # Baseline segment
    ax.text(x[i], base_scaled[i] / 2,
            f"${base_scaled[i]:.0f}M  ({base_scaled[i] / actual_vals[i] * 100:.0f}%)",
            ha="center", va="center", fontsize=7.5, color=CPX_DGREY)
    # Awareness segment
    ax.text(x[i], base_scaled[i] + aw_scaled[i] / 2,
            f"${aw_scaled[i]:.1f}M\n({aw_scaled[i] / actual_vals[i] * 100:.1f}%)",
            ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    # Consideration segment (only if tall enough)
    if co_scaled[i] > 0.4:
        ax.text(x[i], base_scaled[i] + aw_scaled[i] + co_scaled[i] / 2,
                f"${co_scaled[i]:.1f}M\n({co_scaled[i] / actual_vals[i] * 100:.1f}%)",
                ha="center", va="center", fontsize=8, color=CPX_DGREY, fontweight="bold")
    # Actual total at top — bars now reach exactly this height
    ax.text(x[i], actual_vals[i] + 0.5, f"Actual: ${actual_vals[i]:.1f}M",
            ha="center", fontsize=8.5, color=CPX_RED, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(years, fontsize=11)
ax.set_ylabel("Annual Revenue ($M)")
ax.set_title(
    "LBE MMM — Annual Revenue Attribution\n"
    "Segments scaled to actuals  |  Labels = $ attribution + % share of actual annual revenue",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.legend(frameon=False, loc="upper left")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_b6_annual_attribution.png")
plt.show()

# ── ROI: Annual ROI with 95% CI ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
w = 0.35
bars_aw = ax.bar(x - w / 2, aw_roi, width=w, color=CPX_NAVY, alpha=0.85, label="Awareness ROI")
bars_co = ax.bar(x + w / 2, co_roi, width=w, color=CPX_GOLD, alpha=0.85, label="Consideration ROI")
ax.errorbar(x - w / 2, aw_roi, yerr=[aw_roi - aw_roi_lo, aw_roi_hi - aw_roi],
            fmt="none", color="black", capsize=5, lw=1.5)
ax.errorbar(x + w / 2, co_roi, yerr=[co_roi - co_roi_lo, co_roi_hi - co_roi],
            fmt="none", color="black", capsize=5, lw=1.5)
for bar, v in zip(bars_aw, aw_roi):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.08, f"{v:.1f}x",
            ha="center", fontsize=10, fontweight="bold", color=CPX_NAVY)
for bar, v in zip(bars_co, co_roi):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}x",
            ha="center", fontsize=10, fontweight="bold", color=CPX_DGREY)
ax.axhline(1, color=CPX_RED, linestyle="--", linewidth=1.5, label="Break-even (1x)")
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("ROI (Revenue / Spend)")
ax.set_title("LBE MMM — Annual ROI With 95% Credible Intervals",
             fontsize=13, fontweight="bold", color=CPX_NAVY)
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_roi_annual.png")
plt.show()

# ── CF: Counterfactual Lift ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
plot_cf = cf_df.copy()
cf_colors = [CPX_DGREY if s == "Actual" else (CPX_NAVY if d > 0 else CPX_RED)
             for s, d in zip(plot_cf["scenario"], plot_cf["delta_vs_actual_m"])]
ypos = np.arange(len(plot_cf))
ax.barh(ypos, plot_cf["delta_vs_actual_m"], color=cf_colors, alpha=0.85, height=0.6)
ax.axvline(0, color="black", linewidth=1)
for i, (delta, lo_v, hi_v) in enumerate(
    zip(plot_cf["delta_vs_actual_m"], plot_cf["ci_lo_m"] - actual_cf, plot_cf["ci_hi_m"] - actual_cf)
):
    if abs(delta) > 0.3:
        ax.text(delta + (0.2 if delta >= 0 else -0.2), i,
                f"{delta:+.1f}M", va="center",
                ha="left" if delta >= 0 else "right",
                fontsize=9, fontweight="bold", color=CPX_DGREY)
ax.set_yticks(ypos)
ax.set_yticklabels(plot_cf["scenario"], fontsize=10)
ax.set_xlabel("Revenue vs Actual ($M)")
ax.set_title("LBE MMM — Counterfactual Lift vs Actual\n(CI from saved pkl — 80% CI)",
             fontsize=13, fontweight="bold", color=CPX_NAVY)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_cf_counterfactual.png")
plt.show()

summary_business = pd.DataFrame({
    "year": years,
    "actual_revenue_m": np.round(actual_vals, 1),
    "baseline_m": np.round(base_vals, 1),
    "awareness_m": np.round(aw_vals, 1),
    "consideration_m": np.round(co_vals, 1),
    "awareness_roi": np.round(aw_roi, 2),
    "consideration_roi": np.round(co_roi, 2),
})

generated_files = pd.DataFrame([
    {"generated_file": name}
    for name in sorted(os.listdir(EXPORT_DIR))
    if os.path.isfile(os.path.join(EXPORT_DIR, name))
])

# ── Annual Revenue Attribution Table ───────────────────────────────────────────
summary_business = pd.DataFrame({
    "Year":                       years,
    "Actual Revenue ($M)":        np.round(actual_vals,                        1),
    "Baseline — scaled ($M)":     np.round(base_scaled,                        1),
    "Baseline % of Actual":       np.round(base_scaled  / actual_vals * 100,   1),
    "Awareness — scaled ($M)":    np.round(aw_scaled,                          1),
    "Awareness % of Actual":      np.round(aw_scaled    / actual_vals * 100,   1),
    "Consideration — scaled ($M)":np.round(co_scaled,                          1),
    "Consid. % of Actual":        np.round(co_scaled    / actual_vals * 100,   1),
    "Awareness ROI":               np.round(aw_roi, 2),
    "Consideration ROI":           np.round(co_roi, 2),
})

print("Annual Revenue Attribution (contributions scaled to sum to actual):")
display(summary_business)

generated_files = pd.DataFrame([
    {"generated_file": name}
    for name in sorted(os.listdir(EXPORT_DIR))
    if os.path.isfile(os.path.join(EXPORT_DIR, name))
])
display(generated_files)


In [0]:
# ── Helpers ────────────────────────────────────────────────────────────────
import matplotlib.patheffects as pe

def bar_label(ax, x_pos, bottom, height, pct, color, min_height=0.5):
    """Place a % label centred in a bar segment.
    bottom and height are raw $ values; the ax is in $M scale."""
    if abs(height) / 1e6 < min_height:
        return
    light_colors = {CPX_LGREY, CPX_GOLD}
    txt_color = "black" if color in light_colors else "white"
    ax.text(
        x_pos, (bottom + height / 2) / 1e6,
        f"{pct:.0f}%",
        ha="center", va="center", fontsize=11, fontweight="bold", color=txt_color,
        path_effects=[pe.withStroke(linewidth=2,
                      foreground="white" if txt_color == "black" else "black")],
    )

x = np.arange(len(years))   # guard against x being redefined upstream

# ── B6: Annual Revenue Attribution ─────────────────────────────────────────
model_totals = base_vals + aw_vals + co_vals
base_scaled  = base_vals / model_totals * actual_vals
aw_scaled    = aw_vals   / model_totals * actual_vals
co_scaled    = co_vals   / model_totals * actual_vals

# Convert to raw values for bar_label (expects non-divided values)
base_raw = base_scaled * 1e6
aw_raw   = aw_scaled   * 1e6
co_raw   = co_scaled   * 1e6
act_raw  = actual_vals * 1e6

fig, ax = plt.subplots(figsize=(16, 9))
w = 0.5

ax.bar(x, base_scaled,                        width=w, color=CPX_LGREY, alpha=0.95,
       edgecolor="white", linewidth=0.5, label="Baseline (seasonality + weather + trend)")
ax.bar(x, aw_scaled,   bottom=base_scaled,    width=w, color=CPX_NAVY,  alpha=0.90,
       edgecolor="white", linewidth=0.5, label="Awareness")
ax.bar(x, co_scaled,   bottom=base_scaled + aw_scaled, width=w, color=CPX_GOLD, alpha=0.95,
       edgecolor="white", linewidth=0.5, label="Consideration")

# Segment labels using bar_label pattern
for i in range(len(years)):
    bar_label(ax, x[i], 0,       base_raw[i], base_raw[i] / act_raw[i] * 100, CPX_LGREY, min_height=2.0)
    bar_label(ax, x[i], base_raw[i], aw_raw[i],  aw_raw[i]  / act_raw[i] * 100, CPX_NAVY,  min_height=1.0)
    bar_label(ax, x[i], base_raw[i] + aw_raw[i], co_raw[i], co_raw[i]  / act_raw[i] * 100, CPX_GOLD,  min_height=0.5)

# Actual diamond markers — sit exactly on bar tops
ax.scatter(x, actual_vals, color="black", zorder=6, s=70, marker="D", label="Actual")

# Actual $ label — offset above bar top, no collision
for i, act in enumerate(actual_vals):
    ax.annotate(
        f"${act:.1f}M",
        xy=(x[i], act),
        xytext=(x[i], act + actual_vals.max() * 0.025),   # 2.5% of max above bar
        ha="center", fontsize=12, fontweight="bold", color="black",
        arrowprops=dict(arrowstyle="-", color="#AAAAAA", lw=0.8),
    )

ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], fontsize=13)
ax.set_ylabel("Annual Revenue ($M)", fontsize=12)
ax.set_ylim(0, actual_vals.max() * 1.12)   # headroom for top labels
ax.set_title(
    "LBE MMM — Annual Revenue Attribution\n"
    "Bars scaled to actual revenue  |  % = share of annual actual revenue",
    fontsize=15, fontweight="bold", color=CPX_NAVY,
)
ax.legend(frameon=False, loc="upper right", ncol=2, fontsize=11)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_b6_annual_attribution.png")
plt.show()

In [0]:
# ── DIMINISHING RETURNS & MARGINAL RETURN CURVES ─────────────────────────
# Shows total revenue contribution (diminishing returns) and marginal
# revenue per $1 spend for Awareness and Consideration channels.

import matplotlib.ticker as mticker

# ── Posterior mean parameters ──────────────────────────────────────────────
aw_decay_mean = float(draws["aw_decay"].mean())
co_decay_mean = float(draws["co_decay"].mean())
aw_scale_mean = float(draws["aw_scale"].mean())
co_scale_mean = float(draws["co_scale"].mean())
beta_aw_mean  = float(draws["beta_aw"].mean())
beta_co_mean  = float(draws["beta_co"].mean())
intercept_mean = float(draws["intercept"].mean())
beta_trend_mean = float(draws["beta_trend"].mean())
beta_temp_mean  = float(draws["beta_temp"].mean())

# Average baseline context (trend, temp, calendar at their means)
t_avg    = np.asarray(full_data["t"]).mean()
temp_avg = np.asarray(full_data["temp"]).mean()
cal_avg  = np.asarray(full_data["cal"]).mean(axis=0)
bc_mean  = bc.mean(axis=0)
cal_eff_avg = float(cal_avg @ bc_mean)

# Baseline log-revenue (no media)
base_log = intercept_mean + beta_trend_mean * t_avg + beta_temp_mean * temp_avg + cal_eff_avg

# ── Spend range (in raw $ per week, from $0 to 2x observed max) ──────────
# Observed max weekly spend (using proxy: aw_max is the normalizer)
aw_observed_max = aw_max  # max normalized spend = 1, so raw max = aw_max
co_observed_max = co_max

n_pts = 500
aw_spend_range = np.linspace(0, aw_observed_max * 2, n_pts)
co_spend_range = np.linspace(0, co_observed_max * 2, n_pts)

# Normalize spend (model input is spend / max)
aw_norm = aw_spend_range / aw_max
co_norm = co_spend_range / co_max

# ── Saturation: log1p(norm_spend * scale) ─────────────────────────────
aw_sat_curve = np.log1p(aw_norm * aw_scale_mean)
co_sat_curve = np.log1p(co_norm * co_scale_mean)

# ── Revenue contribution: total_rev(with media) - total_rev(no media) ────
def revenue_at_spend(sat_vals, beta, other_sat=0.0):
    """Revenue given saturated media value, holding other channel at 0."""
    log_rev = base_log + beta * sat_vals + other_sat
    return np.exp(log_rev * y_std + y_mean)

baseline_rev = np.exp(base_log * y_std + y_mean)  # revenue with zero media

# Total contribution (diminishing returns curve)
aw_total_rev  = revenue_at_spend(aw_sat_curve, beta_aw_mean) - baseline_rev
co_total_rev  = revenue_at_spend(co_sat_curve, beta_co_mean) - baseline_rev

# ── Marginal return (d(revenue) / d(spend)) via finite differences ───────
aw_marginal = np.gradient(aw_total_rev, aw_spend_range)
co_marginal = np.gradient(co_total_rev, co_spend_range)

# ── Current average weekly spend (vertical reference lines) ──────────────
aw_current_weekly = df_all["Awareness"].mean()
co_current_weekly = df_all["Consideration"].mean()

# ── PLOT 1: Diminishing Returns (Total Response) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.suptitle(
    "LBE MMM \u2014 Diminishing Returns Curves\n"
    "Total weekly revenue contribution vs weekly spend",
    fontsize=14, fontweight="bold", color="black"
)

# Awareness
ax = axes[0]
ax.plot(aw_spend_range / 1e3, aw_total_rev / 1e3, color=CPX_NAVY, linewidth=2.5)
ax.axvline(aw_current_weekly / 1e3, color=CPX_RED, linestyle="--", linewidth=1.5,
           label=f"Current avg: ${aw_current_weekly/1e3:.1f}K/wk")
ax.set_xlabel("Weekly Awareness Spend ($K)")
ax.set_ylabel("Weekly Revenue Contribution ($K)")
ax.set_title("Awareness", fontsize=12, fontweight="bold", color=CPX_NAVY)
ax.legend(frameon=False, fontsize=10)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Consideration
ax = axes[1]
ax.plot(co_spend_range / 1e3, co_total_rev / 1e3, color=CPX_RED, linewidth=2.5)
ax.axvline(co_current_weekly / 1e3, color=CPX_NAVY, linestyle="--", linewidth=1.5,
           label=f"Current avg: ${co_current_weekly/1e3:.1f}K/wk")
ax.set_xlabel("Weekly Consideration Spend ($K)")
ax.set_ylabel("Weekly Revenue Contribution ($K)")
ax.set_title("Consideration", fontsize=12, fontweight="bold", color=CPX_RED)
ax.legend(frameon=False, fontsize=10)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "lbe_diminishing_returns.png")
plt.show()

# ── PLOT 2: Marginal Return (Revenue per $1 additional spend) ───────────
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.suptitle(
    "LBE MMM \u2014 Marginal Return Curves\n"
    "Additional revenue per $1 incremental spend (diminishes with scale)",
    fontsize=14, fontweight="bold", color="black"
)

# Awareness
ax = axes[0]
ax.plot(aw_spend_range / 1e3, aw_marginal, color=CPX_NAVY, linewidth=2.5)
ax.axvline(aw_current_weekly / 1e3, color=CPX_RED, linestyle="--", linewidth=1.5,
           label=f"Current avg: ${aw_current_weekly/1e3:.1f}K/wk")
ax.axhline(1.0, color="#999999", linestyle=":", linewidth=1, label="Break-even ($1 return per $1 spend)")
ax.set_xlabel("Weekly Awareness Spend ($K)")
ax.set_ylabel("Marginal Return ($/$ spend)")
ax.set_title("Awareness", fontsize=12, fontweight="bold", color=CPX_NAVY)
ax.legend(frameon=False, fontsize=10)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Consideration
ax = axes[1]
ax.plot(co_spend_range / 1e3, co_marginal, color=CPX_RED, linewidth=2.5)
ax.axvline(co_current_weekly / 1e3, color=CPX_NAVY, linestyle="--", linewidth=1.5,
           label=f"Current avg: ${co_current_weekly/1e3:.1f}K/wk")
ax.axhline(1.0, color="#999999", linestyle=":", linewidth=1, label="Break-even ($1 return per $1 spend)")
ax.set_xlabel("Weekly Consideration Spend ($K)")
ax.set_ylabel("Marginal Return ($/$ spend)")
ax.set_title("Consideration", fontsize=12, fontweight="bold", color=CPX_RED)
ax.legend(frameon=False, fontsize=10)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "lbe_marginal_returns.png")
plt.show()

# ── Summary: Current marginal ROI at observed spend level ────────────────
aw_idx = np.argmin(np.abs(aw_spend_range - aw_current_weekly))
co_idx = np.argmin(np.abs(co_spend_range - co_current_weekly))

print(f"\nAt current average weekly spend:")
print(f"  Awareness:     ${aw_current_weekly/1e3:.1f}K/wk \u2192 Marginal ROI = {aw_marginal[aw_idx]:.2f}x")
print(f"  Consideration: ${co_current_weekly/1e3:.1f}K/wk \u2192 Marginal ROI = {co_marginal[co_idx]:.2f}x")
print(f"\n  Awareness marginal > 1x means additional $ still profitable")
print(f"  Consideration marginal < 1x means over-saturated at current levels")

In [0]:
# ── Build per-year actual vs predicted ──────────────────────────────────────
df_bias = df_all.copy()
df_bias["predicted"] = full_mean
df_bias["year"] = df_bias["week_date"].dt.year

annual_bias = (
    df_bias.groupby("year")
    .agg(
        actual_m=("LBE_Revenue", lambda x: x.sum() / 1e6),
        predicted_m=("predicted", lambda x: x.sum() / 1e6),
        weeks=("LBE_Revenue", "count"),
    )
    .reset_index()
)
annual_bias["bias_m"]   = annual_bias["predicted_m"] - annual_bias["actual_m"]
annual_bias["bias_pct"] = annual_bias["bias_m"] / annual_bias["actual_m"] * 100
annual_bias["mape_pct"] = (
    df_bias.groupby("year")
    .apply(lambda g: np.mean(np.abs(g["predicted"] - g["LBE_Revenue"]) / g["LBE_Revenue"]) * 100)
    .values
)

print("Annual bias summary:")
display(annual_bias[["year", "actual_m", "predicted_m", "bias_m", "bias_pct", "mape_pct"]].round(2))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.suptitle(
    "LBE MMM — Annual Bias Check",
    fontsize=14, fontweight="bold", color=CPX_NAVY,
)

# Left: actual vs predicted bars
ax = axes[0]
x = np.arange(len(annual_bias))
w = 0.35
bars_act  = ax.bar(x - w / 2, annual_bias["actual_m"],    width=w, color=CPX_DGREY, alpha=0.55, label="Actual")
bars_pred = ax.bar(x + w / 2, annual_bias["predicted_m"], width=w, color=CPX_NAVY,  alpha=0.85, label="Predicted")
for bar, v in zip(bars_act,  annual_bias["actual_m"]):    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.3, f"${v:.0f}M", ha="center", fontsize=8, color=CPX_DGREY)
for bar, v in zip(bars_pred, annual_bias["predicted_m"]): ax.text(bar.get_x() + bar.get_width() / 2, v + 0.3, f"${v:.0f}M", ha="center", fontsize=8, color=CPX_NAVY)
ax.set_xticks(x)
ax.set_xticklabels(annual_bias["year"])
ax.set_ylabel("Annual Revenue ($M)")
ax.set_title("Actual vs Predicted — Annual Totals")
ax.legend(frameon=False)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Right: signed bias % with MAPE annotation
ax = axes[1]
bias_colors = [CPX_NAVY if b >= 0 else CPX_RED for b in annual_bias["bias_pct"]]
bars_bias = ax.bar(x, annual_bias["bias_pct"], color=bias_colors, alpha=0.85, width=0.5)
ax.axhline(0, color="black", linewidth=1)
for bar, pct, bias_m, mape in zip(
    bars_bias, annual_bias["bias_pct"], annual_bias["bias_m"], annual_bias["mape_pct"]
):
    y_label = pct + (0.15 if pct >= 0 else -0.4)
    ax.text(
        bar.get_x() + bar.get_width() / 2, y_label,
        f"{pct:+.1f}%\n({bias_m:+.1f}M)\nMAPE {mape:.1f}%",
        ha="center", va="bottom" if pct >= 0 else "top",
        fontsize=9, color=CPX_NAVY if pct >= 0 else CPX_RED,
    )
ax.set_xticks(x)
ax.set_xticklabels(annual_bias["year"])
ax.set_ylabel("Bias (Predicted − Actual) %")
ax.set_title(
    "Annual Percentage Bias\n+ = over-predicted  |  − = under-predicted",
    color=CPX_NAVY,
)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, "lbe_annual_bias.png")
plt.show()


In [0]:
import seaborn as sns

# ── Assemble baseline features + target ───────────────────────────────────
baseline_cols = ["t", "national_mean_temp"] + cal_cols + ["LBE_Revenue"]

label_map = {
    "t":                   "Trend",
    "national_mean_temp":  "Temperature",
    "is_new_year_wk":      "New Year Wk",
    "is_march_break":      "March Break",
    "is_march_adj":        "March Adj",
    "is_pre_xmas":         "Pre-Xmas",
    "is_summer_peak":      "Summer Peak",
    "is_back_to_school":   "Back to School",
    "is_spring_lull":      "Spring Lull",
    "holiday":             "Holiday",
    "long_weekend":        "Long Weekend",
    "LBE_Revenue":         "LBE Revenue",
}

df_corr = df_all[baseline_cols].rename(columns=label_map)
corr = df_corr.corr()

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))

# Mask upper triangle to avoid redundancy
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# CPX-aligned diverging palette: navy ↔ white ↔ red
cmap = sns.diverging_palette(220, 10, s=85, l=50, as_cmap=True)

ax_hm = sns.heatmap(
    corr,
    mask=mask,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.6,
    linecolor="white",
    square=True,
    annot_kws={"size": 9, "weight": "bold"},
    cbar_kws={"shrink": 0.55, "label": "Pearson r"},
)

# Highlight the LBE Revenue row / column border
n = len(corr)
ax.add_patch(plt.Rectangle((0, n - 1), n, 1, fill=False, edgecolor=CPX_RED, lw=2, clip_on=False))

ax.set_title(
    "LBE MMM — Baseline Feature Correlation Matrix\n"
    "Baseline = Trend + Temperature + Calendar Effects  |  Last row = LBE Revenue (target)",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.tick_params(axis="x", rotation=40, labelsize=10)
ax.tick_params(axis="y", rotation=0,  labelsize=10)

plt.tight_layout()
save_fig(fig, "lbe_baseline_correlation_matrix.png")
plt.show()

# Print top correlates with revenue
rev_corr = corr["LBE Revenue"].drop("LBE Revenue").sort_values(key=abs, ascending=False)
print("Top baseline feature correlations with LBE Revenue:")
display(rev_corr.rename("Pearson r").reset_index().rename(columns={"index": "Feature"}).round(3))


In [0]:
from matplotlib.lines import Line2D

cal_mu_arr  = np.asarray(results["CAL_MU"]).ravel()
cal_sig_arr = np.asarray(results["CAL_SIGMA"]).ravel()

cal_display_labels = {
    "is_new_year_wk":    "New Year Wk",
    "is_march_break":    "March Break",
    "is_march_adj":      "March Adj",
    "is_pre_xmas":       "Pre-Xmas",
    "is_summer_peak":    "Summer Peak",
    "is_back_to_school": "Back to School",
    "is_spring_lull":    "Spring Lull",
    "holiday":           "Holiday",
    "long_weekend":      "Long Weekend",
}
cal_display = [cal_display_labels.get(c, c) for c in cal_cols]

cal_means = bc.mean(axis=0)
cal_lo    = np.percentile(bc, 2.5,  axis=0)
cal_hi    = np.percentile(bc, 97.5, axis=0)
colors_cal = [CPX_NAVY if m > 0 else CPX_RED for m in cal_means]

# ── Plot 5: Calendar Effects ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
bars = ax.bar(cal_display, cal_means, color=colors_cal, alpha=0.82)
ax.errorbar(
    np.arange(len(cal_display)), cal_means,
    yerr=[cal_means - cal_lo, cal_hi - cal_means],
    fmt="none", color=CPX_DGREY, capsize=6, lw=1.8,
)
for i, pm_val in enumerate(cal_mu_arr):
    ax.plot([i - 0.38, i + 0.38], [pm_val, pm_val], color=CPX_GOLD, lw=2, ls="--")
for bar, m, lo, hi in zip(bars, cal_means, cal_lo, cal_hi):
    y_pos = hi + 0.01 if m >= 0 else lo - 0.03
    va    = "bottom" if m >= 0 else "top"
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos, f"{m:+.2f}",
            ha="center", va=va, fontsize=8, color=CPX_NAVY if m >= 0 else CPX_RED)
ax.axhline(0, color="black", lw=0.9)
ax.set_ylabel("Coefficient (log-revenue scale)")
ax.set_title(
    "LBE MMM — Calendar Effects (Posterior)\n"
    "Bars = posterior mean  |  Error bars = 95% CI  |  Dashed = prior mean",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.tick_params(axis="x", rotation=30, labelsize=10)
ax.legend(
    handles=[
        Line2D([0], [0], color=CPX_GOLD, lw=2, ls="--", label="Prior mean"),
        Line2D([0], [0], color=CPX_DGREY, lw=1.8, label="95% CI"),
        Line2D([0], [0], color=CPX_NAVY,  lw=0, marker="s", ms=9, alpha=0.8, label="Positive effect"),
        Line2D([0], [0], color=CPX_RED,   lw=0, marker="s", ms=9, alpha=0.8, label="Negative effect"),
    ],
    frameon=False, loc="upper right",
)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_calendar_effects.png")
plt.show()


In [0]:
# ── Counterfactual lift per calendar event ──────────────────────────────────
# For each event j and each posterior draw i:
#   lift_i_j = sum_{t: cal_j=1}(pred_t * (1 - exp(-bc[i,j] * y_std)))
# pred_t is approximated by full_mean (posterior mean); uncertainty flows
# entirely through the bc draws (8000), which is the dominant source.

df_all["year"] = df_all["week_date"].dt.year   # ensure year column present

cal_lift_m = np.zeros((n_draws, len(cal_cols)))  # $ millions, per draw per event

for j, col in enumerate(cal_cols):
    event_mask  = df_all[col].values.astype(float)          # (T,)
    event_pred  = (full_mean * event_mask).sum()             # total $ in active weeks
    bc_j        = bc[:, j]                                   # (n_draws,)
    lift_factor = 1.0 - np.exp(-bc_j * y_std)               # (n_draws,) — could be -ve
    cal_lift_m[:, j] = event_pred * lift_factor / 1e6       # $M per draw

cal_lift_mean = cal_lift_m.mean(axis=0)
cal_lift_lo   = np.percentile(cal_lift_m,  2.5, axis=0)
cal_lift_hi   = np.percentile(cal_lift_m, 97.5, axis=0)

# Number of active weeks and avg weekly lift
event_weeks_n  = np.array([int(df_all[col].sum()) for col in cal_cols])
wkly_lift_mean = np.where(event_weeks_n > 0, cal_lift_mean / event_weeks_n, 0.0)

# ── Per-year breakdown ───────────────────────────────────────────────────────
years_uniq = sorted(df_all["year"].unique())
yr_lift_mean = {}     # {year: array shape (n_cal,)}
for yr in years_uniq:
    yr_mask = (df_all["year"] == yr).values.astype(float)
    yr_row  = np.zeros(len(cal_cols))
    for j, col in enumerate(cal_cols):
        ev_mask_yr  = df_all[col].values.astype(float) * yr_mask
        ev_pred_yr  = (full_mean * ev_mask_yr).sum()
        yr_row[j]   = ev_pred_yr * (1 - np.exp(-bc[:, j] * y_std)).mean() / 1e6
    yr_lift_mean[yr] = yr_row

# ── Summary table ────────────────────────────────────────────────────────────
cal_summary = pd.DataFrame({
    "Calendar Event":     cal_display,
    "Weeks Active":       event_weeks_n,
    "Total Lift $M":      np.round(cal_lift_mean, 2),
    "95% CI Lo $M":       np.round(cal_lift_lo,   2),
    "95% CI Hi $M":       np.round(cal_lift_hi,   2),
    "Avg per Week $M":    np.round(wkly_lift_mean, 2),
    **{f"{yr} $M": np.round(yr_lift_mean[yr], 2) for yr in years_uniq},
}).sort_values("Total Lift $M", ascending=False, key=abs).reset_index(drop=True)

print("Calendar event revenue contribution (3-year total, 95% CI from 8,000 posterior draws):")
display(cal_summary)

# ── Chart: total 3-year contribution, sorted by magnitude ───────────────────
sort_idx      = np.argsort(cal_lift_mean)[::-1]
s_display     = [cal_display[i]   for i in sort_idx]
s_means       = cal_lift_mean[sort_idx]
s_lo          = cal_lift_lo[sort_idx]
s_hi          = cal_lift_hi[sort_idx]
s_wkly        = wkly_lift_mean[sort_idx]
s_colors      = [CPX_NAVY if m >= 0 else CPX_RED for m in s_means]

x = np.arange(len(cal_cols))

# ── Chart 1: 3-year total contribution with 95% CI ──────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
bars = ax.bar(x, s_means, color=s_colors, alpha=0.85, width=0.6)
ax.errorbar(
    x, s_means,
    yerr=[s_means - s_lo, s_hi - s_means],
    fmt="none", color=CPX_DGREY, capsize=6, lw=1.8,
)
for bar, m, hi_v, lo_v in zip(bars, s_means, s_hi, s_lo):
    y_pos = hi_v + 0.05 if m >= 0 else lo_v - 0.15
    ax.text(
        bar.get_x() + bar.get_width() / 2, y_pos,
        f"${m:+.1f}M",
        ha="center", va="bottom" if m >= 0 else "top",
        fontsize=9, fontweight="bold",
        color=CPX_NAVY if m >= 0 else CPX_RED,
    )
ax.axhline(0, color="black", lw=0.9)
ax.set_xticks(x)
ax.set_xticklabels(s_display, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Revenue Contribution ($M, 3-year total)")
ax.set_title(
    "LBE MMM — Calendar Event Revenue Contribution (3-Year Total)\n"
    "Bars = posterior mean  |  Error bars = 95% CI  |  Sorted by magnitude",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_calendar_revenue_contribution_total.png")
plt.show()

# ── Chart 2: average weekly lift per event ─────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
bars2 = ax.bar(x, s_wkly, color=s_colors, alpha=0.85, width=0.6)
for bar, m in zip(bars2, s_wkly):
    if abs(m) > 0.01:
        y_pos = m + (0.005 if m >= 0 else -0.015)
        ax.text(
            bar.get_x() + bar.get_width() / 2, y_pos,
            f"${m * 1e3:.0f}K/wk",
            ha="center", va="bottom" if m >= 0 else "top",
            fontsize=9, fontweight="bold",
            color=CPX_NAVY if m >= 0 else CPX_RED,
        )
ax.axhline(0, color="black", lw=0.9)
ax.set_xticks(x)
ax.set_xticklabels(s_display, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Average Revenue Lift per Active Week ($M)")
ax.set_title(
    "LBE MMM — Calendar Event Average Weekly Lift\n"
    "Total lift ÷ weeks active  |  Sorted by 3-year magnitude",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "lbe_calendar_revenue_contribution_weekly.png")
plt.show()


In [0]:
import matplotlib.patches as mpatches

# ── Model-derived constants from posterior ────────────────────────────────
AW_DECAY_MEAN  = float(np.asarray(draws["aw_decay"]).mean())
LAG_MIN        = 4
LAG_MAX        = 6
CPX_GREEN      = "#2E7D32"
CPX_MID        = "#AABBCC"

# ── Weekly-of-year aggregation from df_all ───────────────────────────────
df_flight       = df_all.copy()
df_flight["woy"] = df_flight["week_date"].dt.isocalendar().week.astype(int)

woy_avg = (
    df_flight.groupby("woy")
    .agg(
        revenue      =("LBE_Revenue",   "mean"),
        awareness    =("Awareness",     "mean"),
        consideration=("Consideration", "mean"),
    )
    .reset_index()
)
wks = woy_avg["woy"].values

# ── Calendar zone definitions ───────────────────────────────────────────────
PEAK_WKS = [1, 11, 33, 49, 50]

optimal_aw_wks = set()
for p in PEAK_WKS:
    for lag in range(LAG_MIN, LAG_MAX + 1):
        w = p - lag
        if w <= 0:
            w += 52
        optimal_aw_wks.add(w)
optimal_aw_wks = sorted(optimal_aw_wks)

TROUGH_WKS   = list(range(19, 24)) + list(range(36, 42))
OPT_SPANS    = [(5, 7), (27, 29), (43, 48)]
TROUGH_SPANS = [(19, 23), (36, 41)]
LAG_ARROWS   = [(6, 11, "~5 wk lag"), (28, 33, "~5 wk lag"), (45, 50, "~5 wk lag"), (48, 1, "wraps →\nJan")]
peak_labels  = {1: "New Year\nWeek", 11: "March\nBreak", 33: "Summer\nPeak", 49: "Pre-\nXmas"}

aw_total  = woy_avg["awareness"].sum()
aw_opt    = woy_avg.loc[woy_avg["woy"].isin(optimal_aw_wks),  "awareness"].sum()
aw_trough = woy_avg.loc[woy_avg["woy"].isin(TROUGH_WKS),      "awareness"].sum()
aw_late   = woy_avg.loc[woy_avg["woy"].isin(PEAK_WKS),        "awareness"].sum()
aw_other  = aw_total - aw_opt - aw_trough - aw_late

co_arr     = woy_avg["consideration"].values
rv_arr     = woy_avg["revenue"].values
corr_lag0  = np.corrcoef(co_arr, rv_arr)[0, 1]
corr_lag4  = np.corrcoef(np.roll(co_arr, 4), rv_arr)[0, 1]

print(f"Awareness in optimal windows : {aw_opt/aw_total*100:.1f}%")
print(f"Awareness in trough periods  : {aw_trough/aw_total*100:.1f}%")
print(f"Awareness in peak week itself: {aw_late/aw_total*100:.1f}%")
print(f"Awareness in neutral weeks   : {aw_other/aw_total*100:.1f}%")

# ── Shared helpers ────────────────────────────────────────────────────────────
XTICKS      = [1, 5, 9, 13, 17, 21, 26, 30, 35, 39, 43, 48, 52]
XTICKLABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec",""]

def shade_zones(ax, trough=True, optimal=True):
    if trough:
        for i, (s, e) in enumerate(TROUGH_SPANS):
            ax.axvspan(s, e, alpha=0.08, color=CPX_RED,
                       label="Revenue trough (avoid spend)" if i == 0 else "")
    if optimal:
        for i, (s, e) in enumerate(OPT_SPANS):
            ax.axvspan(s, e, alpha=0.13, color=CPX_GREEN,
                       label="Optimal Awareness window" if i == 0 else "")

def clean_ax(ax):
    ax.set_xlim(1, 52)
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICKLABELS)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

FOOTER = (
    f"Note: Weekly averages across {df_flight['year'].min()}–{df_flight['year'].max()}. "
    f"Optimal windows = {LAG_MIN}–{LAG_MAX} weeks before each revenue peak. "
    f"Posterior mean Awareness decay = {AW_DECAY_MEAN:.3f}."
)

# ── Panel 1: Revenue calendar ──────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(16, 9))
shade_zones(ax1)
ax1.plot(wks, woy_avg["revenue"] / 1e6, color=CPX_NAVY, lw=2.5, zorder=4,
         label="Avg weekly revenue ($M)")
ax1.fill_between(wks, woy_avg["revenue"] / 1e6, alpha=0.08, color=CPX_NAVY)

for wk, lbl in peak_labels.items():
    rv = woy_avg.loc[woy_avg["woy"] == wk, "revenue"].values
    if len(rv):
        ax1.annotate(
            lbl, xy=(wk, rv[0] / 1e6),
            xytext=(wk, rv[0] / 1e6 * 0.93 + 0.2),
            ha="center", fontsize=9, color=CPX_GOLD, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=CPX_GOLD, lw=1),
        )

arrow_y = woy_avg["revenue"].max() / 1e6 * 0.70
for opt_ctr, peak_wk, lbl in LAG_ARROWS:
    ax1.annotate(
        "", xy=(peak_wk, arrow_y), xytext=(opt_ctr, arrow_y),
        arrowprops=dict(arrowstyle="->", color=CPX_GREEN, lw=1.5, linestyle="dashed"),
    )
    ax1.text((opt_ctr + peak_wk) / 2, arrow_y + 0.08, lbl,
             ha="center", fontsize=9, color=CPX_GREEN, style="italic")

ax1.set_ylabel("Revenue ($M)", fontsize=11)
ax1.set_title(
    f"LBE Media Flighting — Panel 1: Revenue Calendar\n"
    f"Avg weekly LBE revenue by week-of-year ({df_flight['year'].min()}–{df_flight['year'].max()})  "
    f"|  Green bands = optimal Awareness spend windows (4–6 wks before peak)",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax1.legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.9)
clean_ax(ax1)
fig1.text(0.5, -0.02, FOOTER, ha="center", fontsize=8, color=CPX_DGREY, style="italic")
plt.tight_layout()
save_fig(fig1, "lbe_flighting_p1_revenue_calendar.png")
plt.show()

# ── Panel 2: Awareness spend by zone ──────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(16, 9))

bar_colors = []
for w in wks:
    if w in TROUGH_WKS:       bar_colors.append(CPX_RED)
    elif w in PEAK_WKS:       bar_colors.append(CPX_GOLD)
    elif w in optimal_aw_wks: bar_colors.append(CPX_GREEN)
    else:                      bar_colors.append(CPX_MID)

ax2.bar(wks, woy_avg["awareness"] / 1e3, color=bar_colors, alpha=0.85, width=0.85)
shade_zones(ax2)

ymax = woy_avg["awareness"].max() / 1e3
for lbl, xpos, color in [
    (f"Optimal\n{aw_opt/aw_total*100:.0f}% of spend",             6,  CPX_GREEN),
    (f"Trough\n{aw_trough/aw_total*100:.0f}% of spend",           38, CPX_RED),
    (f"Peak wk (too late)\n{aw_late/aw_total*100:.0f}% of spend", 49, CPX_GOLD),
]:
    ax2.text(xpos, ymax * 0.88, lbl, ha="center", fontsize=9, fontweight="bold", color=color,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=color, alpha=0.92))

ax2.set_ylabel("Avg Weekly Spend ($K)", fontsize=11)
ax2.set_title(
    "LBE Media Flighting — Panel 2: Awareness Spend by Zone\n"
    "Colour = whether spend lands in the right window relative to revenue peaks",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax2.legend(
    handles=[
        mpatches.Patch(color=CPX_GREEN, alpha=0.85, label=f"Optimal window — {aw_opt/aw_total*100:.0f}% of spend"),
        mpatches.Patch(color=CPX_RED,   alpha=0.85, label=f"Revenue trough — {aw_trough/aw_total*100:.0f}% (low return)"),
        mpatches.Patch(color=CPX_GOLD,  alpha=0.85, label=f"Peak week itself — {aw_late/aw_total*100:.0f}% (too late for lag)"),
        mpatches.Patch(color=CPX_MID,   alpha=0.85, label=f"Neutral weeks — {aw_other/aw_total*100:.0f}% of spend"),
    ],
    loc="upper left", fontsize=10, frameon=True, framealpha=0.9, ncol=2,
)
clean_ax(ax2)
fig2.text(0.5, -0.02, FOOTER, ha="center", fontsize=8, color=CPX_DGREY, style="italic")
plt.tight_layout()
save_fig(fig2, "lbe_flighting_p2_awareness_spend.png")
plt.show()

# ── Panel 3: Consideration spend vs revenue ────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(16, 9))
ax3b = ax3.twinx()

ax3.fill_between(wks, woy_avg["revenue"] / 1e6, alpha=0.07, color=CPX_NAVY)
ax3.plot(wks, woy_avg["revenue"] / 1e6, color=CPX_NAVY, lw=1.5, alpha=0.5,
         label="Avg weekly revenue (left axis)")

co_colors = [CPX_RED if w in TROUGH_WKS else CPX_GOLD for w in wks]
ax3b.bar(wks, woy_avg["consideration"] / 1e3, color=co_colors, alpha=0.75, width=0.85,
         label="Consideration spend (right axis)")
shade_zones(ax3, optimal=False)

ax3.text(
    0.02, 0.96,
    f"Correlation — Consideration spend vs Revenue:\n"
    f"  Same week (lag 0):      r = {corr_lag0:+.2f}\n"
    f"  4 weeks later (lag 4): r = {corr_lag4:+.2f}\n"
    f"  → No consistent lead or lag signal at any horizon",
    transform=ax3.transAxes, fontsize=10, color=CPX_DGREY,
    va="top", ha="left", style="italic",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#FFFBF0", edgecolor="#DDDDDD", alpha=0.92),
)

ax3.set_ylabel("Revenue ($M)", color=CPX_NAVY, fontsize=11)
ax3b.set_ylabel("Avg Weekly Spend ($K)", color=CPX_GOLD, fontsize=11)
ax3.set_title(
    "LBE Media Flighting — Panel 3: Consideration Spend vs Revenue Rhythm\n"
    "No consistent timing relationship — spend does not lead or follow revenue peaks",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2,
           loc="lower right", fontsize=10, frameon=True, framealpha=0.9)
ax3.set_xlim(1, 52); ax3.set_xticks(XTICKS); ax3.set_xticklabels(XTICKLABELS)
ax3.grid(False)
ax3.spines["top"].set_visible(False)
ax3b.spines["top"].set_visible(False)
ax3b.spines["right"].set_color("#CCCCCC")
fig3.text(0.5, -0.02, FOOTER, ha="center", fontsize=8, color=CPX_DGREY, style="italic")
plt.tight_layout()
save_fig(fig3, "lbe_flighting_p3_consideration_vs_revenue.png")
plt.show()

# ── Summary stats ────────────────────────────────────────────────────────────────
flighting_summary = pd.DataFrame({
    "Zone": ["Optimal windows", "Revenue troughs", "Peak week itself", "Neutral weeks"],
    "Weeks": [
        len(optimal_aw_wks), len(TROUGH_WKS), len(PEAK_WKS),
        52 - len(optimal_aw_wks) - len(TROUGH_WKS) - len(PEAK_WKS),
    ],
    "Awareness Spend $K (avg/yr)": np.round(
        [aw_opt, aw_trough, aw_late, aw_other], 1
    ),
    "Share of Awareness %": np.round(
        [aw_opt/aw_total*100, aw_trough/aw_total*100, aw_late/aw_total*100, aw_other/aw_total*100], 1
    ),
})
print(f"\nPosterior mean Awareness decay: {AW_DECAY_MEAN:.3f}  (lag {LAG_MIN}–{LAG_MAX} wks)")
print(f"Consideration vs revenue correlation — lag 0: {corr_lag0:+.3f}  |  lag 4: {corr_lag4:+.3f}")
display(flighting_summary)
